# Single-Topic Document Q&A with RAG

###Jade Webb

###Feb 2026

###References:
###RAG: https://www.kaggle.com/code/markishere/day-2-document-q-a-with-rag/notebook
###Dataset: https://www.kaggle.com/datasets/samuelmatsuoharris/single-topic-rag-evaluation-dataset/data

##Installations and Imports

In [1]:
pip install google-genai

In [2]:
pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found

In [3]:
pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 2.3 MB/s eta 0:00:00


In [4]:
import getpass
import os
import pandas as pd
import chromadb
import ast

from google import genai
from google.genai import types
from IPython.display import Markdown
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry
from google.genai import types

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [5]:
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your GOOGLE API key: ")

Enter your GOOGLE API key: ··········


In [6]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OPENAI API key: ")

Enter your OPENAI API key: ··········


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Load Gemini Client

In [8]:
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

In [9]:
#view available models
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/aqa
models/

##Load and Inspect 3 Q&A Datasets

Single Passage: 40 Q&A pairs whose answers must be generated by analyzing a single passage in a single document.

Multi Passage: 40 Q&A pairs whose answers must be generated by analyzing multiple passages in a single document.

No Answer: 40 questions that do not have an answer within the documents.

In [10]:
#single passage
file_path = '/content/drive/MyDrive/Colab Notebooks/RAG/single_passage_answer_questions.csv'
single_df = pd.read_csv(file_path)

In [11]:
single_df.head()

,document_index,question,answer
0,0,What do keybullet kin drop?,Keybullet kin drop a key upon death.
1,0,What kind of gun does the bandana bullet kin use?,The bandana bullet kin wields a machine pistol.
2,1,What do the giants look like?,"One giant is burly, grey-skinned, and 20 feet ..."
3,1,What happens on day 2?,"After a few miles of winding tunnel, you emerg..."
4,2,What were the requirements for the project?,The tool had the following requirements:\n- Ch...


In [61]:
#replace inacccurate question
single_df.loc[2] = [1, "What creatures are encountered on day 4 and what do they want?", "A horde of kobolds are encountered. They demand food in the ambush."]
#clarify poor questions
single_df.loc[4] = [2, "What were the requirements for the STICI-Note project?", "The project had the following requirements:\n- Chatbot that you can ask questions and get answers in response (conversational memory is not required).\n- Information is taken from an unstructured text file.\n- It must be able to tell me if it doesn’t know the answer to my question.\n- Fast.\n- Efficient enough to run on my MacBook with other programs without any performance issues.\n- Locally run for privacy and to ensure it will always be free, runnable, and consistent."]
single_df.loc[5] = [2, "What data was used to test the STICI-Note prototype?", "Grace Hopper's Wikipedia page and Alan Turing's Wikipedia page were used to test the prototype."]
single_df.loc[6] = [3, "How do the data storage options for enterprise RAG Pipelines compare?", "For fast start: use SQLite3 and ChromaDB (File-based) out-of-the-box - no install required.\nFor speed + scale: use MongoDB (text collection) and Milvus (vector db) - install with Docker Compose\nFor postgres: use Postgres for both text collection and vector DB - install with Docker Compose\nFor mix-and-match: LLMWare supports 3 text collection databases (Mongo, Postgres, SQLite) and 10 vector databases (Milvus, PGVector-Postgres, Neo4j, Redis, Mongo-Atlas, Qdrant, Faiss, LanceDB, ChromaDB and Pinecone)."]
single_df.loc[10] = [5, "What are the key topics of the article about Data Science Impact?", "The key topics of this article are: \"why prioritizing impact matters not just for managers, but also ICs\"; \"why focusing on impact is hard\"; \"how to maximize your impact\"; and \"how to overcome common challenges in driving real impact\"."]
single_df.loc[35] = [17, "What year is Alan Wake 2 set?", "Alan Wake 2 is set in 2023, thirteen years after the events of Alan Wake."]
single_df.loc[37] = [18, "Who was resurrected with a group of other murder victims in the book?", "Lou was resurrected along with a handful of other women murdered by a single serial killer."]
single_df.loc[38] = [19, "What categories does Huang's paper split the RAG technique paradigm into?", "This paper organizes the RAG paradigm into four categories: pre-retrieval, retrieval, post-retrieval, and generation."]
single_df.loc[39] = [19, "In what way can including unrelated documents affect the RAG system in Huang's research?", "The inclusion of irrelevant documents can significantly improve accuracy."]

In [14]:
single_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   document_index  40 non-null     int64 
 1   question        40 non-null     object
 2   answer          40 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.1+ KB


In [15]:
single_df.isnull().sum()

,0
document_index,0
question,0
answer,0


In [16]:
single_df.duplicated().sum()

np.int64(0)

In [17]:
#multi passage
file_path = '/content/drive/MyDrive/Colab Notebooks/RAG/multi_passage_answer_questions.csv'
multi_df = pd.read_csv(file_path)

In [18]:
multi_df.head()

,document_index,question,answer
0,0,Which enemy types wield an AK-47?,Assault-rifle wielding Bullet and Tankers wiel...
1,0,What makes jammed enemies different?,"Jammed Keybullet Kin drop 2 keys instead of 1,..."
2,1,What enemies are encountered in the second enc...,26 kobolds and 1 kobold inventor are encounter...
3,1,What monsters are encountered in this journey?,"Ropers, kobolds, kobold inventors, fire giants..."
4,2,What framework was chosen to execute the RAG p...,The LangChain framework was used to orchestrat...


In [19]:
#replace inacccurate question
multi_df.loc[3] = [1, "What smells are encountered in this journey?", "Smells of hard water, wet earth, and a rancid odor are encountered along the journey."]

In [20]:
multi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   document_index  40 non-null     int64 
 1   question        40 non-null     object
 2   answer          40 non-null     object
dtypes: int64(1), object(2)
memory usage: 1.1+ KB


In [21]:
multi_df.isnull().sum()

,0
document_index,0
question,0
answer,0


In [22]:
multi_df.duplicated().sum()

np.int64(0)

In [23]:
#no answer
file_path = '/content/drive/MyDrive/Colab Notebooks/RAG/no_answer_questions.csv'
no_df = pd.read_csv(file_path)

In [24]:
no_df.head()

,document_index,question
0,0,How much health does the Mutant Bullet Kin have?
1,0,Where can bishops be found?
2,1,What happened on day 10?
3,1,What did the goblins say?
4,2,Why was the H100 GPU chosen for computation?


In [74]:
#clarify poor questions
no_df.loc[0] = [0, "How much health (numerical quantity) does the Mutant Bullet Kin have?"]
no_df.loc[29] = [14, "Who is Perry and Jackie's cousin?"]
no_df.loc[38] = [19, "Which paper by Hinton et. al. was used in Huang's paper?"]
no_df.loc[39] = [19, "Why was hardware such a limiting factor in Huang's research?"]

In [26]:
no_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   document_index  40 non-null     int64 
 1   question        40 non-null     object
dtypes: int64(1), object(1)
memory usage: 772.0+ bytes


In [27]:
no_df.isnull().sum()

,0
document_index,0
question,0


In [28]:
no_df.duplicated().sum()

np.int64(0)

##Load and Inspect Documents Dataset

20 niche text documents (articles, blogs, documentation) produced within the last few years (as of July 2024), ranging from a few thousand to a few ten thousand words.

In [29]:
file_path = '/content/drive/MyDrive/Colab Notebooks/RAG/single_topic_rag_evaluation_documents.csv'
df = pd.read_csv(file_path)

In [30]:
df.head()

,index,source_url,text
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   index       20 non-null     int64 
 1   source_url  20 non-null     object
 2   text        20 non-null     object
dtypes: int64(1), object(2)
memory usage: 612.0+ bytes


In [32]:
df.isnull().sum()

,0
index,0
source_url,0
text,0


In [33]:
df.duplicated().sum()

np.int64(0)

## Embedding Database
Perform document embedding using Gemini Embedding 001 and populate a Chroma database.

In [34]:
#retry embedding when per-minute limit is reached
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})

In [35]:
#generate document or query embeddings with Gemini API
class GeminiEmbeddingFunction(EmbeddingFunction):

    document_mode = True

    @retry.Retry(predicate=is_retriable)
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:  #generate embeddings for documents
            embedding_task = "retrieval_document"
        else:                   #generate embeddings for queries
            embedding_task = "retrieval_query"

        response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=input,
            config=types.EmbedContentConfig(task_type=embedding_task),
        )
        return [e.values for e in response.embeddings]

In [36]:
embed_func = GeminiEmbeddingFunction()
#document mode
embed_func.document_mode = True

/tmp/ipython-input-652/1492628580.py:1: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  embed_func = GeminiEmbeddingFunction()


In [37]:
chromadb_client = chromadb.Client()
db = chromadb_client.get_or_create_collection(name="single_topic_db", embedding_function=embed_func)

db.add(documents=list(df['text']), ids=[str(i) for i in range(len(df['text']))])

In [38]:
db.count()

20

##Retrieval and Generation
Perform RAG for all 3 Q&A datasets with Gemini 3 Flash.

In [77]:
single_queries = list(single_df['question'])
multi_queries = list(multi_df['question'])
no_queries = list(no_df['question'])

In [79]:
def RAG(queries):
  embed_func.document_mode = False
  answers = []
  for query in queries:
    result = db.query(query_texts=[query], n_results=2) #get 2 most relevant documents
    [all_passages] = result["documents"]

    prompt = f"""You are a helpful and informative bot that answers questions by only using text from the reference passage(s) included below.
    Do not rely on any other information.
    You must respond in complete sentences and be concise, including only the most relevant direct answer from the passage(s).
    If a passage is irrelevant to the answer, you may ignore it.
    The user has no knowledge of your use of passages or the passages themselves.
    Never mention passages in your response.
    If the answer is not found in the passages, your response must be limited to exactly the phrase 'I don't know'.
    Interpret each question strictly and literally.
    Each response must be specific. If you cannot produce a specific answer, say 'I don't know'.
    Never hallucinate, make up an answer, or make assumptions.

    QUESTION: {query}
    """

    for passage in all_passages:
      passage_oneline = passage.replace("\n", " ")
      prompt += f"PASSAGE: {passage_oneline}\n"

    answer = client.models.generate_content(
      model="gemini-3-flash-preview",
      contents=prompt)

    answers.append((answer.text))

  return answers


In [67]:
single_answers = RAG(single_queries)
single_answers

['Keybullet Kin drop a key upon death, while Jammed Keybullet Kin drop two keys.',
 'The Bandana Bullet Kin wields Machine Pistols.',
 'On day 4, twenty-six kobolds and one kobold inventor are encountered. These creatures want food and items, threatening to stab the group if they do not drop their things.',
 'On day 2, travelers emerge in a small grotto of stalactites and stalagmites and encounter two Ropers.',
 'The requirements for the STICI-Note project included a chatbot that can answer questions from an unstructured text file and state if it does not know an answer. Additionally, the tool needed to be fast, locally run for privacy and consistency, and efficient enough to run on a MacBook without performance issues.',
 'The STICI-Note prototype was tested using the Wikipedia pages of Grace Hopper and Alan Turing.',
 'Data storage options for enterprise RAG pipelines include SQLite3 and ChromaDB for file-based quick starts, MongoDB and Milvus for higher speed and scale, and Postgres

In [42]:
multi_answers = RAG(multi_queries)
multi_answers

['Assault-rifle wielding Bullet Kin and Tankers are the enemy types that wield an AK-47.',
 'Jammed enemies run faster, teleport away more quickly, have the potential to drop twice the loot or keys, and can deal contact damage in cases where their normal counterparts do not.',
 'The second encounter consists of twenty-six Kobolds and one Kobold Inventor.',
 'In this journey, you encounter the smell of hard water and wet earth, a rancid smell in a dirt and rock chamber, and a stink caused by a large pile of nappies and methane.',
 'LangChain was the chosen framework to orchestrate the RAG process, while LlamaIndex, LitGPT, and llmware were considered as alternatives.',
 'The shortlisted large language models for this project include tinyllama-1.1b-chat-v1.0 Q6_K, Phi 3 Q4_K_M, bartowski/dolphin-2.8-experiment26-7b-GGUF Q3_K_L, mgonzs13/Mistroll-7B-v2.2-GGU, and QuantFactory/Meta-Llama-3-8B-Instruct Q3_K_M. The vector databases that were shortlisted are Chroma, Qdrant, and Vespa.',
 "I d

In [80]:
no_answers = RAG(no_queries)
no_answers

["I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know.",
 "I don't know."]

##LLM-As-A-Judge
Use GPT-4o to judge Gemini 3 Flash's RAG performance for all 3 Q&A datasets.

For the single and multi passage datasets, correct answers receive a score of 1.

For the no answer dataset, responses of "I don't know" receive a score of 1.

Each dataset receives a final score: the total of all answer scores (maximum score = 40).

In [44]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

In [81]:
judge_prompt = '''\
You are a helpful and informative bot that judges the accuracy of a list of answers, based on an answer key.
You will be given a list of answers and an answer key, and judge whether the answers are correct.
Evaluate every answer exactly once.
The answers and the answer key are in the same order, so compare them against each other in the same order, one-by-one.
Do not compare the answers to the answer key word-for-word, because due to the nature of textual answers, variations are natural.
Your task is to provide exactly one score for each answer, that represents the answer's quality on a binary scale of 0 and 1.
0 means that the answer contains wrong information or states only "I don't know".
1 means that the answer contains information from the answer key.
Your rating must be an integer.
Do not penalize the answer for providing information beyond what is supplied in the answer key.

ANSWERS: {answers}
ANSWER KEY: {answer_key}

Generate a list containing all {length} scores, in order of evaluation.
Do not produce anything besides the requested list.
'''

In [46]:
scenario_chain = ChatPromptTemplate.from_template(judge_prompt) | llm | StrOutputParser()

In [47]:
def judge(answers, answer_key):
  return scenario_chain.invoke(
      {
        "answers": answers,
        "answer_key": answer_key,
        "length": len(answers)
      }
  )

In [69]:
single_scores = judge(single_answers, list(single_df['answer']))
single_scores

'[1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]'

In [82]:
single_scores_list = ast.literal_eval(single_scores)
sum(single_scores_list)

37

In [50]:
multi_scores = judge(multi_answers, list(multi_df['answer']))
multi_scores

'[1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]'

In [83]:
multi_scores_list = ast.literal_eval(multi_scores)
sum(multi_scores_list)

35

In [84]:
no_scores = judge(no_answers, ["I don't know"]*40)
no_scores

'[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]'

In [85]:
no_scores_list = ast.literal_eval(no_scores)
sum(no_scores_list)

40

##Results
Create a scores table and determine what queries were answered incorrectly.

In [89]:
results_data = {
    "Q&A Dataset": ["Single Passage", "Multi Passage", "No Answer"],
    "Final Score/40": [sum(single_scores_list), sum(multi_scores_list), sum(no_scores_list)]
}

results = pd.DataFrame(results_data)
results

,Q&A Dataset,Final Score/40
0,Single Passage,37
1,Multi Passage,35
2,No Answer,40


In [97]:
def incorrect_answers(scores, queries):
  index = 0
  queries_with_incorrect_answers = []
  for score in scores:
    if (score == 0):
      queries_with_incorrect_answers.append(queries[index])
    index += 1
  return queries_with_incorrect_answers

In [98]:
incorrect_answers(single_scores_list, single_queries)

['When was UTF-8 support added for European languages?',
 'How do I make a button?',
 'What kinds of AI carry "systematic risks"?']

In [99]:
incorrect_answers(multi_scores_list, multi_queries)

['What kind of model is the bling-phi-3 model',
 'What forms of exercise did we do?',
 'Where is broadcasting used?',
 'What is Karpathy known for?',
 'What things does Scratch do?']